# padding-amount-formula-convT — faded example 2: inverse-solve padding from target size

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `padding-amount-formula-convT`. Running the beacon reports progress on the `CNN: ConvT padding amount formula` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: ConvT padding amount formula` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`padding-amount-formula-convT`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "padding-amount-formula-convT"
DD_SUBTOPIC = "CNN: ConvT padding amount formula"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Inverting `H_out = (H_in - 1)*S - 2P + K` for `P` gives `P = ((H_in - 1)*S + K - H_out) // 2`, valid only when the gap is non-negative and even. This is how decoder padding args are back-solved from a chosen output size.

## Faded exercise 2

Complete `convT_padding_for` by computing the gap `(H_in-1)*S + K - H_out` from which P is derived. Fill in the gap expression.

**Fill in:** the gap expression (h_in-1)*s + k - h_out_target used to solve for padding

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(6)

def convT_padding_for(h_in, k, s, h_out_target):
    gap = None  # TODO: the gap expression (h_in-1)*s + k - h_out_target used to solve for padding
    if gap < 0 or gap % 2 != 0:
        return None
    return gap // 2

print(convT_padding_for(8, 4, 2, 16))


def _test():
    t.manual_seed(66)
    # solvable cases: P recovered must hit target on the real module
    for h_in, k, s, target in [(4, 3, 1, 6), (4, 3, 1, 4), (8, 4, 2, 18), (10, 3, 1, 10)]:
        p = convT_padding_for(h_in, k, s, target)
        assert p is not None and p >= 0
        layer = nn.ConvTranspose2d(1, 1, kernel_size=k, stride=s, padding=p)
        actual = layer(t.randn(1, 1, h_in, h_in)).shape[-1]
        assert actual == target, (h_in, k, s, target, p, actual)
    # unsolvable: target exceeds no-pad max -> None
    assert convT_padding_for(4, 3, 1, 100) is None
    # parity violation -> None (gap odd)
    assert convT_padding_for(4, 4, 1, 6) is None


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

t.manual_seed(6)

def convT_padding_for(h_in, k, s, h_out_target):
    gap = (h_in - 1) * s + k - h_out_target
    if gap < 0 or gap % 2 != 0:
        return None
    return gap // 2

print(convT_padding_for(8, 4, 2, 16))
```
</details>